# D2 · XGBoost importances: gain, weight, cover, permutation and contributions

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

On a dataset with a **very informative** feature, a moderately informative one, two correlated ones and two noise ones (one continuous and one binary) compare five XGBoost importance measures: `weight` (how many times used), `gain` (average gain), `cover`, permutation on the test set and the mean of the **per-example contributions** (`pred_contribs`).
Check that the contributions **sum to the raw margin** and explain a single prediction.

## Dataset

`generate()` (given): 4000 examples with `strong`, `medium`, `corr_a`, `corr_b` (two noisy versions of the same factor, both informative), `noise_c` (continuous) and `noise_b` (binary). `X_tr, X_te, y_tr, y_te`; model `XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=4)` already trained (`model`).

## Educational purpose

Know what each "importance" measures: `weight` rewards continuous features (many thresholds), `gain` is more sensible but depends on the training set, the permutation on new data answers the question "is it really needed?", the per-example contributions are a **local explanation** that sums to the margin.

## Task

1. Build `table` (index: the six features; columns `weight`, `gain`, `cover`, `permutation`, `contributions`) with `booster.get_score(importance_type=...)` for the first three (0 if the feature does not appear), the permutation on the test set (AUC, `n_repeats=10`, `random_state=0`) and the mean of the absolute values of `pred_contribs` on the test set. Normalize each column to sum 1.
2. Answer: `weight_rewards_noise` (True if `noise_c` has a share of at least 10% in `weight` while having less than 2% in the permutation), `gain_relegates_noise` (True if in `gain` `noise_c` has less than half the share of `medium`) and `top_by_method` (dictionary method → feature with the largest share).
3. **Additivity:** for `X_te` compute `contribs = booster.predict(DMatrix(X_te), pred_contribs=True)` and `margin = booster.predict(DMatrix(X_te), output_margin=True)`; save `sum_ok` (True if `contribs.sum(1)` matches `margin` with tolerance 1e-4) and `rebuilt_prob` (sigmoid of the margin) compared with `predict_proba` (`prob_ok`).
4. **Explain an example:** for example `i = 0` of the test set save `explanation` (`pd.Series` of the contributions of the six features sorted by absolute value, plus `bias`) and `decisive_feature` (name of the feature with the largest absolute contribution).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def generate(n=4000, seed=0):
    rng = np.random.default_rng(seed)
    strong, medium, z = rng.normal(size=(3, n))
    corr_a, corr_b = z + rng.normal(0, 0.3, n), z + rng.normal(0, 0.3, n)
    y = (2 * strong + 0.7 * medium + 1.2 * z + rng.normal(0, 1, n) > 0).astype(int)
    X = pd.DataFrame({"strong": strong, "medium": medium, "corr_a": corr_a, "corr_b": corr_b, "noise_c": rng.normal(size=n), "noise_b": rng.integers(0, 2, n).astype(float)})
    return X, y


X, y = generate()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
model = xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
booster = model.get_booster()
names = list(X.columns)

## Your work

In [ ]:
table = pd.DataFrame(index=names, columns=["weight", "gain", "cover", "permutation", "contributions"], dtype=float)
table

In [ ]:
weight_rewards_noise = None
gain_relegates_noise = None
top_by_method = None

In [ ]:
sum_ok = None
prob_ok = None

In [ ]:
explanation = None
decisive_feature = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`scores = booster.get_score(importance_type="gain")` returns a dictionary only for the features used: `pd.Series(scores).reindex(names).fillna(0)`. Contributions: `xgb.DMatrix(X_te)` (with the column names of the DataFrame); the last column is the bias. Sigmoid: `1 / (1 + np.exp(-margin))`. Permutation: `permutation_importance(model, X_te, y_te, n_repeats=10, random_state=0, scoring="roc_auc").importances_mean`.'):
    check(table.notna().all().all() and close(table.sum().to_numpy(), np.ones(5)), "complete table with columns normalized to sum 1", "divide each column by its sum")
    check(weight_rewards_noise is True and gain_relegates_noise is True, "the continuous noise weighs over 10% for weight, less than half of `medium` for gain and less than 2% for permutation", "weight counts the cuts: many distinct values, many cuts")
    check(top_by_method["gain"] == "strong" and top_by_method["permutation"] == "strong" and top_by_method["contributions"] == "strong", "gain, permutation and contributions put `strong` first", "idxmax per column")
    check(sum_ok is True and prob_ok is True, "the contributions sum to the raw margin; the sigmoid of the margin reproduces predict_proba", "pred_contribs has a final 'bias' column")
    check(len(explanation) == 7 and explanation.index[-1] == "bias" or "bias" in explanation.index, "the explanation includes six features and the constant term", "one column per feature and one for the bias")
    check(decisive_feature in names and abs(explanation[decisive_feature]) == explanation.drop("bias").abs().max(), "the decisive feature is the one with the largest absolute contribution", "argmax of the absolute contributions")
done()

---
The commented solution is in `solutions/D_interpret/D2_xgboost_importances_gain_weight_and_contributions.ipynb`. Open it after you pass the check, or after you have really tried.